# Graph Traversal Retrieval [Step 4 - Retrieving by Walking, Not by Matching]

> **MLCourse - Agentic AI - Advanced RAG - Graph RAG**

A graph is only useful for RAG once it can answer a natural-language question.
That needs three pieces, and this notebook builds each:

1. **Entity linking** - find the nodes the question is about.
2. **Traversal** - walk out from those nodes to collect relevant facts.
3. **Verbalisation** - turn the collected subgraph into text the LLM can read.

The result is a retriever with a completely different failure profile from
vector search: it is exact where it works, and it returns nothing at all when
entity linking misses.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]
print("paragraphs:", len(paragraphs))

paragraphs: 237


### Extracted with the LLM in notebook 02 and saved here so notebooks 03-05 do not


In [ ]:
# each re-pay the extraction cost. In a real system this is exactly what you do:
# extraction is a ONE-OFF indexing step whose output is persisted.
TRIPLES = [
    ("Alice", "follows", "White Rabbit"),
    ("Alice", "falls down", "rabbit hole"),
    ("White Rabbit", "carries", "pocket watch"),
    ("White Rabbit", "serves", "Duchess"),
    ("White Rabbit", "is herald for", "King of Hearts"),
    ("Alice", "drinks from", "little bottle"),
    ("little bottle", "causes", "shrinking"),
    ("Alice", "eats", "cake"),
    ("cake", "causes", "growing"),
    ("Alice", "meets", "Caterpillar"),
    ("Caterpillar", "sits on", "mushroom"),
    ("mushroom", "causes", "size change"),
    ("Caterpillar", "advises", "Alice"),
    ("Alice", "meets", "Cheshire Cat"),
    ("Cheshire Cat", "belongs to", "Duchess"),
    ("Cheshire Cat", "vanishes leaving", "grin"),
    ("Cheshire Cat", "directs Alice to", "Mad Hatter"),
    ("Alice", "attends", "mad tea party"),
    ("Mad Hatter", "attends", "mad tea party"),
    ("March Hare", "attends", "mad tea party"),
    ("Dormouse", "attends", "mad tea party"),
    ("Mad Hatter", "quarrelled with", "Time"),
    ("Duchess", "nurses", "baby"),
    ("baby", "turns into", "pig"),
    ("Duchess", "employs", "Cook"),
    ("Cook", "throws", "pepper"),
    ("Alice", "meets", "Queen of Hearts"),
    ("Queen of Hearts", "orders", "beheadings"),
    ("Queen of Hearts", "plays", "croquet"),
    ("croquet", "uses", "flamingo"),
    ("croquet", "uses", "hedgehog"),
    ("Queen of Hearts", "is married to", "King of Hearts"),
    ("Queen of Hearts", "commands", "playing cards"),
    ("playing cards", "paint", "white roses"),
    ("Knave of Hearts", "is accused of stealing", "tarts"),
    ("Queen of Hearts", "baked", "tarts"),
    ("King of Hearts", "presides over", "trial"),
    ("Knave of Hearts", "stands at", "trial"),
    ("Mad Hatter", "testifies at", "trial"),
    ("Alice", "testifies at", "trial"),
    ("Gryphon", "takes Alice to", "Mock Turtle"),
    ("Queen of Hearts", "sends", "Gryphon"),
    ("Mock Turtle", "tells", "his history"),
    ("Mock Turtle", "dances", "Lobster Quadrille"),
    ("Gryphon", "dances", "Lobster Quadrille"),
]

print(len(TRIPLES), "curated (subject, relation, object) triples")


In [5]:
import networkx as nx

G = nx.MultiDiGraph()
for s, r, o in TRIPLES:
    G.add_edge(s, o, relation=r)

print("graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

graph: 38 nodes, 45 edges


### 2. Entity linking - the step everything depends on

If you cannot map "the cat" in the question to the `Cheshire Cat` node, nothing
downstream can work. Three approaches, increasing in cost and robustness:

- **String matching** - fast, free, brittle. Misses paraphrase entirely.
- **Embedding similarity** over node names - handles paraphrase, needs a
  threshold, can link confidently to the wrong thing.
- **LLM extraction** - ask the model which of the known entities the question is
  about. Most robust, costs a call.

We build all three and compare them on the same questions, because seeing them
disagree is the fastest way to understand the tradeoff.

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

NODES = sorted(G.nodes())
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
node_vectors = encoder.encode(NODES, normalize_embeddings=True)


def link_by_string(question):
    low = question.lower()
    return [n for n in NODES if n.lower() in low]


def link_by_embedding(question, threshold=0.45, top_n=4):
    sims = node_vectors @ encoder.encode([question], normalize_embeddings=True)[0]
    order = np.argsort(sims)[::-1][:top_n]
    return [(NODES[i], float(sims[i])) for i in order if sims[i] >= threshold]


def link_by_llm(question):
    out = ask(
        "Which of these entities is the question about? Reply with the matching "
        "entity names exactly as written, comma-separated, or NONE.\n\n"
        f"Entities: {', '.join(NODES)}\n\nQuestion: {question}"
    )
    return [n for n in NODES if n.lower() in out.lower()]


QUESTIONS = [
    "Who owns the cat that directed Alice to the Hatter?",
    "What did the Queen order at the croquet game?",
    "Which characters were at both the tea party and the trial?",
]

for q in QUESTIONS:
    print("Q:", q)
    print("  string   :", link_by_string(q))
    print("  embedding:", [f"{n} ({s:.2f})" for n, s in link_by_embedding(q)])
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Q: Who owns the cat that directed Alice to the Hatter?
  string   : ['Alice']
  embedding: ['Alice (0.55)']

Q: What did the Queen order at the croquet game?
  string   : ['croquet']
  embedding: ['croquet (0.64)']

Q: Which characters were at both the tea party and the trial?
  string   : ['trial']
  embedding: ['mad tea party (0.51)', 'trial (0.48)']



In [7]:
q = QUESTIONS[0]
print("Q:", q)
print("  llm      :", link_by_llm(q))
print("\nNote where the cheap methods fail: 'the cat' and 'the Hatter' are not "
      "literal node names, so string matching under-links.")

Q: Who owns the cat that directed Alice to the Hatter?


  llm      : ['Cheshire Cat']

Note where the cheap methods fail: 'the cat' and 'the Hatter' are not literal node names, so string matching under-links.


### 3. Traversal - collecting the relevant subgraph

Once you have seed nodes, retrieval means **expanding outward**. The radius is
the key parameter, and it behaves very differently from `k` in vector search:

- **1 hop** - direct facts about the entity. Precise, often too thin.
- **2 hops** - facts about the entity and its immediate relations. **The usual
  sweet spot**, and enough for most multi-hop questions.
- **3+ hops** - in a small, well-connected graph this reaches nearly everything,
  which is the same as retrieving nothing in particular.

That last point is the graph equivalent of context bloat, and it arrives much
faster than people expect.

In [8]:
def subgraph_around(seeds, hops=2):
    """Everything within `hops` steps of any seed node."""
    und = G.to_undirected(as_view=False)
    keep = set()
    for seed in seeds:
        if seed in und:
            keep |= set(nx.ego_graph(und, seed, radius=hops).nodes())
    return G.subgraph(keep).copy()


for hops in [1, 2, 3]:
    sg = subgraph_around(["Cheshire Cat"], hops)
    print(f"{hops} hop(s) from Cheshire Cat: "
          f"{sg.number_of_nodes():>2} nodes, {sg.number_of_edges():>2} edges "
          f"({sg.number_of_nodes() / G.number_of_nodes():.0%} of the graph)")

1 hop(s) from Cheshire Cat:  5 nodes,  4 edges (13% of the graph)
2 hop(s) from Cheshire Cat: 16 nodes, 19 edges (42% of the graph)
3 hop(s) from Cheshire Cat: 31 nodes, 37 edges (82% of the graph)


### 4. Verbalisation - turning edges into sentences

LLMs read text, not adjacency lists. Each edge becomes one short sentence. Keep
it terse: a subgraph of 40 edges is 40 lines, and every line costs context.

In [9]:
def verbalise(graph, limit=60):
    lines = [f"{u} {d['relation']} {v}." for u, v, d in graph.edges(data=True)]
    return "\n".join(sorted(set(lines))[:limit])


sg = subgraph_around(["Cheshire Cat", "Mad Hatter"], hops=2)
print(f"subgraph: {sg.number_of_nodes()} nodes, {sg.number_of_edges()} edges\n")
print(verbalise(sg))

subgraph: 20 nodes, 25 edges

Alice attends mad tea party.
Alice drinks from little bottle.
Alice eats cake.
Alice falls down rabbit hole.
Alice follows White Rabbit.
Alice meets Caterpillar.
Alice meets Cheshire Cat.
Alice meets Queen of Hearts.
Alice testifies at trial.
Caterpillar advises Alice.
Cheshire Cat belongs to Duchess.
Cheshire Cat directs Alice to Mad Hatter.
Cheshire Cat vanishes leaving grin.
Dormouse attends mad tea party.
Duchess employs Cook.
Duchess nurses baby.
King of Hearts presides over trial.
Knave of Hearts stands at trial.
Mad Hatter attends mad tea party.
Mad Hatter quarrelled with Time.
Mad Hatter testifies at trial.
March Hare attends mad tea party.
Queen of Hearts is married to King of Hearts.
White Rabbit is herald for King of Hearts.
White Rabbit serves Duchess.


### 5. The complete graph retriever

Link, traverse, verbalise, generate. Notice there is no similarity score
anywhere in this pipeline - retrieval is structural.

In [10]:
def graph_rag(question, hops=2, use_llm_linking=True):
    seeds = link_by_llm(question) if use_llm_linking else link_by_string(question)
    if not seeds:
        seeds = [n for n, _ in link_by_embedding(question, threshold=0.35, top_n=3)]
    sg = subgraph_around(seeds, hops=hops)
    facts = verbalise(sg)
    answer = ask(
        "You are given facts from a knowledge graph, one per line. Answer the "
        "question using ONLY these facts. Show the chain of facts you used. If "
        "the facts are insufficient, say exactly which link is missing.\n\n"
        f"Facts:\n{facts}\n\nQuestion: {question}\nAnswer:"
    )
    return seeds, sg, answer


seeds, sg, answer = graph_rag("Who owns the cat that directed Alice to the Hatter?")
print("linked entities:", seeds)
print(f"subgraph: {sg.number_of_nodes()} nodes, {sg.number_of_edges()} edges")
print("\nanswer:")
print(answer)

linked entities: ['Cheshire Cat']
subgraph: 16 nodes, 19 edges

answer:
**Chain of facts:**
1.  **Cheshire Cat directs Alice to Mad Hatter.** (Identifies the cat that directed Alice to the Hatter as the Cheshire Cat)
2.  **Cheshire Cat belongs to Duchess.** (Identifies the owner of the Cheshire Cat as the Duchess)

**Answer:**
Duchess


In [11]:
seeds, sg, answer = graph_rag(
    "Which characters were present at both the mad tea party and the trial?")
print("linked entities:", seeds)
print(f"subgraph: {sg.number_of_nodes()} nodes, {sg.number_of_edges()} edges")
print("\nanswer:")
print(answer)

linked entities: ['Alice', 'King of Hearts', 'Queen of Hearts']
subgraph: 31 nodes, 38 edges

answer:
To determine which characters were present at both the mad tea party and the trial, I will analyze the provided facts to identify attendees for each event and find the intersection.

**Step 1: Identify characters present at the mad tea party.**
Scanning the facts for "attends mad tea party":
1. `Alice attends mad tea party.`
2. `Dormouse attends mad tea party.`
3. `Mad Hatter attends mad tea party.`
4. `March Hare attends mad tea party.`

Characters at the mad tea party: **Alice, Dormouse, Mad Hatter, March Hare**.

**Step 2: Identify characters present at the trial.**
Scanning the facts for "testifies at trial", "stands at trial", or "presides over trial":
1. `Alice testifies at trial.`
2. `King of Hearts presides over trial.`
3. `Knave of Hearts stands at trial.`
4. `Mad Hatter testifies at trial.`

Characters at the trial: **Alice, King of Hearts, Knave of Hearts, Mad Hatter**.

**S

### 6. Path-constrained retrieval - the sharpest tool here

For "how is X related to Y" questions, do not gather a neighbourhood at all.
Compute the **shortest path** and verbalise only that. The context is tiny and
the answer is exact.

In [12]:
def path_facts(source, target):
    und = G.to_undirected(as_view=False)
    try:
        nodes = nx.shortest_path(und, source, target)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None, None
    lines = []
    for a, b in zip(nodes, nodes[1:]):
        rel = next((d["relation"] for x, y, d in G.edges(data=True)
                    if {x, y} == {a, b}), "is related to")
        lines.append(f"{a} {rel} {b}.")
    return nodes, "\n".join(lines)


nodes, facts = path_facts("Queen of Hearts", "Mock Turtle")
print("path:", " -> ".join(nodes))
print("\nfacts given to the LLM (that is ALL of them):")
print(facts)

print("\nanswer:")
print(ask(
    "Using ONLY these facts, explain in two sentences how the two entities are "
    f"connected.\n\nFacts:\n{facts}\n\n"
    "Question: What connects the Queen of Hearts to the Mock Turtle?\nAnswer:"
))

path: Queen of Hearts -> Gryphon -> Mock Turtle

facts given to the LLM (that is ALL of them):
Queen of Hearts sends Gryphon.
Gryphon takes Alice to Mock Turtle.

answer:


The Queen of Hearts sends the Gryphon, who then takes Alice to the Mock Turtle. This sequence of actions establishes the Gryphon as the intermediary connecting the Queen of Hearts to the Mock Turtle.


Compare that to notebook 01, where the same question produced three plausible
paragraphs and no answer. Here the context is four lines, and the chain is
explicit and checkable.

### 7. Where graph retrieval fails

Be equally clear about the failure modes - they are sharper than vector search's.

- **Entity linking miss = total failure.** No seed node, no subgraph, no answer.
  Vector search degrades gracefully; graph retrieval falls off a cliff.
- **A missing edge is invisible.** If extraction never produced
  `Cheshire Cat --belongs to--> Duchess`, the question is unanswerable and
  nothing tells you why.
- **Content questions are answered badly.** Ask "what did the Caterpillar
  actually say to Alice?" and the graph offers `Caterpillar advises Alice` - true,
  and useless. The prose is gone.
- **Hop explosion.** In a dense graph, 3 hops is the whole graph.

Let us watch failure mode three happen, because it is the one that motivates the
next notebook.

In [13]:
q = "What exactly did the Caterpillar say to Alice about her size?"
seeds, sg, answer = graph_rag(q, hops=2)
print("Q:", q)
print("linked:", seeds)
print("\ngraph-only answer:")
print(answer)
print("\n^ The graph knows THAT the Caterpillar advised Alice. It does not know "
      "WHAT was said - that lives in the prose, which extraction discarded.")

Q: What exactly did the Caterpillar say to Alice about her size?
linked: ['Alice', 'Caterpillar']

graph-only answer:
The provided facts state that "Caterpillar advises Alice" and "mushroom causes size change," but they do not contain the specific dialogue or exact words the Caterpillar said to Alice regarding her size.

Missing link: The specific statement or advice content from the Caterpillar to Alice about size.

^ The graph knows THAT the Caterpillar advised Alice. It does not know WHAT was said - that lives in the prose, which extraction discarded.


### 8. Key takeaways

- Graph retrieval is **link -> traverse -> verbalise -> generate**, with no
  similarity score anywhere.
- **Entity linking is the weak point**; LLM linking is the most robust and worth
  its cost.
- **2 hops** is the usual sweet spot; 3+ typically returns the whole graph.
- **Path-constrained retrieval** gives tiny, exact context for relational
  questions - the clearest win over vector search in this whole module.
- The graph cannot answer *content* questions, which is precisely why the next
  notebook combines it with vector search.

Next: [`05_hybrid_graph_vector_rag.ipynb`](05_hybrid_graph_vector_rag.ipynb).